# Day 8 — Model Improvement & Confidence Logic

In [1]:
import pandas as pd
import joblib

dataset = pd.read_csv(
    "../data/processed/labeled_dataset.csv"
)

model = joblib.load(
    "../models/xgb_baseline.pkl"
)

dataset.head()

,file,mean,std,rms,max,min,peak_to_peak,health_stage
0,2003.10.22.12.06.24,-0.094593,0.081124,0.124614,0.388,-0.720,1.108,Healthy
1,2003.10.22.12.09.13,-0.094903,0.079517,0.123811,0.388,-0.654,1.042,Healthy
2,2003.10.22.12.14.13,-0.096187,0.080219,0.125246,0.317,-0.623,0.940,Healthy
3,2003.10.22.12.19.13,-0.095613,0.080827,0.125197,0.457,-0.598,1.055,Healthy
4,2003.10.22.12.24.13,-0.095133,0.082036,0.125618,0.388,-0.623,1.011,Healthy


In [2]:
from sklearn.preprocessing import LabelEncoder

X = dataset.drop(
    columns=["file","health_stage"]
)

encoder = LabelEncoder()

y = encoder.fit_transform(
    dataset["health_stage"]
)

In [3]:
prob = model.predict_proba(X)

prob[:5]

array([[5.2369363e-03, 5.4695275e-02, 9.3522227e-01, 4.8455759e-03],
       [4.0157302e-03, 3.4828894e-02, 9.5691186e-01, 4.2435220e-03],
       [8.4565324e-04, 2.9117400e-03, 9.9546331e-01, 7.7935541e-04],
       [1.9090323e-03, 4.8328396e-03, 9.9228811e-01, 9.7003358e-04],
       [1.5224770e-03, 2.1309493e-02, 9.7544855e-01, 1.7194716e-03]],
      dtype=float32)

In [4]:
import numpy as np

confidence = np.max(
    prob,
    axis=1
)

confidence = (
    confidence*100
).round(2)

confidence[:10]

array([93.52, 95.69, 99.55, 99.23, 97.54, 99.3 , 57.96, 99.61, 84.17,
       94.67], dtype=float32)

In [6]:
pred = model.predict(X)

labels = encoder.inverse_transform(
    pred
)

results = pd.DataFrame({
    "prediction":labels,
    "confidence":confidence
})

results.head(10)

,prediction,confidence
0,Healthy,93.519997
1,Healthy,95.690002
2,Healthy,99.550003
3,Healthy,99.230003
4,Healthy,97.540001
5,Healthy,99.300003
6,Early_Degradation,57.959999
7,Healthy,99.610001
8,Healthy,84.169998
9,Healthy,94.669998


In [8]:
dataset["prediction"]=labels

dataset["confidence"]=confidence

dataset.head(10)

,file,mean,std,rms,max,min,peak_to_peak,health_stage,prediction,confidence
0,2003.10.22.12.06.24,-0.094593,0.081124,0.124614,0.388,-0.720,1.108,Healthy,Healthy,93.519997
1,2003.10.22.12.09.13,-0.094903,0.079517,0.123811,0.388,-0.654,1.042,Healthy,Healthy,95.690002
2,2003.10.22.12.14.13,-0.096187,0.080219,0.125246,0.317,-0.623,0.940,Healthy,Healthy,99.550003
3,2003.10.22.12.19.13,-0.095613,0.080827,0.125197,0.457,-0.598,1.055,Healthy,Healthy,99.230003
4,2003.10.22.12.24.13,-0.095133,0.082036,0.125618,0.388,-0.623,1.011,Healthy,Healthy,97.540001
5,2003.10.22.12.29.13,-0.095882,0.081731,0.125988,0.432,-0.564,0.996,Healthy,Healthy,99.300003
6,2003.10.22.12.34.13,-0.095573,0.081315,0.125483,0.439,-0.667,1.106,Healthy,Early_Degradation,57.959999
7,2003.10.22.12.39.13,-0.095952,0.080231,0.125074,0.381,-0.603,0.984,Healthy,Healthy,99.610001
8,2003.10.22.12.44.13,-0.095882,0.080815,0.125396,0.413,-0.696,1.109,Healthy,Healthy,84.169998
9,2003.10.22.12.49.13,-0.094768,0.081248,0.124828,0.479,-0.625,1.104,Healthy,Healthy,94.669998


In [9]:
dataset.to_csv(
    "../data/processed/prediction_results.csv",
    index=False
)

In [10]:
def maintenance_action(stage, confidence):

    if stage=="Healthy":
        return "No action required"

    elif stage=="Early_Degradation":
        return "Schedule inspection"

    elif stage=="Critical":
        return "Maintenance within 7 days"

    else:
        return "Immediate shutdown"


dataset["maintenance_action"] = [
    maintenance_action(
        s,
        c
    )
    for s,c in zip(
        dataset["prediction"],
        dataset["confidence"]
    )
]

dataset[
[
"prediction",
"confidence",
"maintenance_action"
]
].head(10)

,prediction,confidence,maintenance_action
0,Healthy,93.519997,No action required
1,Healthy,95.690002,No action required
2,Healthy,99.550003,No action required
3,Healthy,99.230003,No action required
4,Healthy,97.540001,No action required
5,Healthy,99.300003,No action required
6,Early_Degradation,57.959999,Schedule inspection
7,Healthy,99.610001,No action required
8,Healthy,84.169998,No action required
9,Healthy,94.669998,No action required


In [11]:
dataset.to_csv(
    "../data/processed/prediction_results.csv",
    index=False
)